# データベース演習 第3回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

## インデックスの作成と効果の確認

### 参考：データベースへのインデックスの付与と性能改善の確認

In [ ]:
# 使用する環境
!pip install jupysql --quiet
from sqlalchemy import create_engine, text
import sqlite3
import time
import pandas as pd
import random
import string
import numpy as np

In [ ]:
# ちょっとおまじない．jupysqlとpythonと両方からアクセスできるようにする

engine = create_engine('sqlite:///:memory:')

In [ ]:
# jupysqlでメモリ上のSQLiteデータベースにアクセスする
%load_ext sql
%sql engine

In [ ]:
# テーブルの作成
%%sql

CREATE TABLE ユーザー (
    ユーザーID INTEGER PRIMARY KEY,
    名前 TEXT,
    年齢 INTEGER
);

In [ ]:
# テーブルを確認（データは入っていない）
%%sql

SELECT * FROM ユーザー;


In [ ]:
# 100万件のサンプルデータを作成する
# プログラムの中身は理解できなくてかまわない
with engine.connect() as conn:

  # ランダムデータ生成（100万件）

  # ランダムな名前を作成する関数
  def random_name():
      return ''.join(random.choices(string.ascii_lowercase, k=10))

  # ランダムなデータを作成
  data = [(i, random_name(), random.randint(18, 90)) for i in range(1_000_000)]

  # データをユーザーテーブルに挿入
  conn.execute(text('BEGIN'))  # トランザクション開始
  conn.execute(text('INSERT INTO ユーザー (ユーザーID, 名前, 年齢) VALUES (:id, :name, :age)'), [
      {'id': i, 'name': name, 'age': age} for (i, name, age) in data
  ])
  conn.execute(text('COMMIT'))

In [ ]:
# 作成されたデータを確認する
%%sql

SELECT * FROM ユーザー limit 10;

In [ ]:
# 作成されたデータの件数を確認する
%%sql

SELECT count(*) FROM ユーザー;

In [ ]:
# 性能測定用の関数を定義する
def measure_query(conn, sql, params=()):
    start = time.time()
    result = conn.execute(sql, params)
    rows = result.fetchall()
    elapsed = time.time() - start
    print(f"{elapsed:.4f}秒で{len(rows)}件取得できた")

In [ ]:
# インデックスなしでの検索時間を計測

# SQL文
sql = 'SELECT * FROM ユーザー WHERE 年齢 = 30'

with engine.connect() as conn:
    measure_query(conn, text(sql))


In [ ]:
# 実行プランを確認（インデックスなし）
# → "SCAN ユーザー" のように表示されれば，テーブル全体を順スキャンしている
%%sql
EXPLAIN QUERY PLAN SELECT * FROM ユーザー WHERE 年齢 = 30;

In [ ]:
# 年齢にインデックスを追加
%%sql
CREATE INDEX idx_年齢 ON ユーザー(年齢)

In [ ]:
# 実行プランを確認（インデックスあり）
# → "SEARCH ユーザー USING INDEX idx_年齢" のように表示されれば，インデックスが使われている
%%sql
EXPLAIN QUERY PLAN SELECT * FROM ユーザー WHERE 年齢 = 30;

In [ ]:
# インデックスありでの検索時間を計測

# SQL文
sql = 'SELECT * FROM ユーザー WHERE 年齢 = 30'

with engine.connect() as conn:
    measure_query(conn, text(sql))

### 演習：インデックスの効果確認

- 3つの列を持つテーブルを作成します。
- 各列は整数型で、次の特徴を持つデータを格納します：
  - **一様なデータ**：0〜9999までの値をランダムに一様分布から生成します。
  - **偏ったデータ**：指数分布を使って、値に偏りを持たせたデータを生成します。
  - **少数種類のデータ**：0〜4の5種類のみの値を持つデータを生成します。
- テーブルとデータは、こちらが用意したセルを実行することで自動的に作成されます。

---

## 課題内容

- **インデックスがない場合**と**インデックスを作成した場合**で、各列に対する検索クエリの性能（実行時間）を比較してください。
- 性能測定には、参考として示した `measure_query` 関数を使用してください。
  - 測定セルの例と同様に使用できます。

- **検索条件に指定する値**は以下とします：
  - 一様なデータ：5000
  - 偏ったデータ：10
  - 少数種類のデータ：0

---
## 課題の提出

- 作業結果の残ったこのノートブックをリンク提出してください
- 3つのうち一番改善効果が大きかった列を，WebClassで答えてください．

In [ ]:
# テーブルの作成
%%sql

CREATE TABLE サンプル (
    番号 INTEGER PRIMARY KEY,
    一様なデータ INTEGER,
    偏ったデータ INTEGER,
    少数種類のデータ INTEGER
);

In [ ]:
# データの作成：中身が理解できていなくてかまいません

np.random.seed(42)
n = 100000  # レコード数

# 各列のデータを作成
uniform_data = np.random.randint(0, 10000, size=n)
skewed_data = np.random.exponential(scale=50, size=n).astype(int)
few_unique_data = np.random.choice([0, 1, 2, 3, 4], size=n, p=[0.7, 0.1, 0.1, 0.05, 0.05])

# 4. データをリスト化
data = list(zip(
    range(1, n + 1),
    map(int, uniform_data),
    map(int, skewed_data),
    map(int, few_unique_data)
))

# 5. データ挿入
with engine.connect() as conn:
    conn.execute(text('BEGIN'))  # トランザクション開始

    conn.execute(
        text('''
            INSERT INTO サンプル (番号, 一様なデータ, 偏ったデータ, 少数種類のデータ)
            VALUES (:番号, :一様なデータ, :偏ったデータ, :少数種類のデータ)
        '''),
        [
            {'番号': no, '一様なデータ': uni, '偏ったデータ': skew, '少数種類のデータ': few}
            for (no, uni, skew, few) in data
        ]
    )

    conn.execute(text('COMMIT'))  # トランザクション終了


In [ ]:
# データを確認する
%%sql

SELECT * FROM サンプル LIMIT 10;

#### ここから自力でお願いします！：
* どの列がインデックスによる改善効果が大きいかを調べてください
* 必要に応じて `EXPLAIN QUERY PLAN` を使い，インデックスが使われているか確認すること